# DTD Hong Kong Trading-Day Sequence Test

This notebook calls `vanke_dtd.dtd` to process one Temporary Input date at a time and writes successful results to an independent Temporary Output file.

Core rules:

1. Only dates with `HK_Open=True` in `China_HK_Trading_Calendar.xlsx` can produce a DTD.
2. A Hong Kong market closure returns `SKIPPED_HK_CLOSED` and does not write Output.
3. An open-market date must be the next Hong Kong trading day after the last completed DTD date.
4. Duplicate dates, skipped open days, insufficient calendar coverage, and missing target-date Input raise an error.
5. Temporary Output always has the columns `Comp_no`, `Date`, and `DTD`.

## 1 Parameters for real files

The default assumptions are:

- `runtime/demo_workspace/vanke.xlsx` stores this demo's confirmed Input and Output copy.
- `runtime/demo_workspace/vanke_dtd_temporary_data.xlsx` stores incremental Input.
- `runtime/demo_workspace/temporary_output.xlsx` stores provisional output.
- Enter one date at a time in `TARGET_DATE`, for example `20251215`.

In [ ]:
from pathlib import Path
import sys
import sys

import pandas as pd

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_DIR / 'src'))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from vanke_dtd.dtd import (
    DTDTestError,
    MissingTradingDayError,
    process_daily_dtd,
    read_temporary_output,
)

WORKSPACE_DIR = PROJECT_DIR / "runtime" / "demo_workspace"
CONFIRMED_FILE = WORKSPACE_DIR / "vanke.xlsx"
CALENDAR_FILE = PROJECT_DIR / "data" / "controlled" / "China_HK_Trading_Calendar.xlsx"
TEMPORARY_INPUT_FILE = WORKSPACE_DIR / "vanke_dtd_temporary_data.xlsx"
TEMPORARY_OUTPUT_FILE = WORKSPACE_DIR / "temporary_output.xlsx"

TARGET_DATE = "20251215"
COMPANY = None
RUN_REAL_FILE = False  # Set True only after confirming file paths and TARGET_DATE.

## 2 Run one date

In [ ]:
if RUN_REAL_FILE:
    daily_result = process_daily_dtd(
        TARGET_DATE,
        confirmed_file=CONFIRMED_FILE,
        temporary_input_file=TEMPORARY_INPUT_FILE,
        temporary_output_file=TEMPORARY_OUTPUT_FILE,
        calendar_file=CALENDAR_FILE,
        company=COMPANY,
    )
    display(pd.Series(daily_result.as_dict(), name="value"))
    display(read_temporary_output(TEMPORARY_OUTPUT_FILE))
else:
    print("RUN_REAL_FILE=False: this cell did not modify any real Temporary file.")

## 3 Isolated example test

The example below creates test files in a system temporary directory and does not modify project files. The example Input contains 20251213 through 20251217; market values and rates are test assumptions only.

The sequence is: 13, 14, 15, deliberately request 17, then run 16, and finally rerun 17.

In [ ]:
from tempfile import TemporaryDirectory

import numpy as np


def make_demo_input(output_path: Path) -> None:
    confirmed = pd.read_excel(CONFIRMED_FILE, sheet_name="Input")
    last = confirmed.tail(1).copy()
    assumptions = {
        20251213: (61977.573739, 2.46),
        20251214: (61977.573739, 2.46),
        20251215: (62000.0, 2.45),
        20251216: (61500.0, 2.44),
        20251217: (61000.0, 2.43),
    }
    rows = []
    for date, (market_cap, risk_free_rate) in assumptions.items():
        row = last.copy()
        row["Date"] = date
        row["CUR_MKT_CAP(HKD)"] = market_cap
        row["Risk_Free_Rate"] = risk_free_rate
        rows.append(row)
    pd.concat(rows, ignore_index=True).to_excel(output_path, sheet_name="Input", index=False)


test_rows = []
with TemporaryDirectory() as test_directory:
    test_directory = Path(test_directory)
    demo_input = test_directory / "data_temporary.xlsx"
    demo_output = test_directory / "temporary_output.xlsx"
    make_demo_input(demo_input)

    for date in ("20251213", "20251214", "20251215"):
        result = process_daily_dtd(
            date,
            confirmed_file=CONFIRMED_FILE,
            temporary_input_file=demo_input,
            temporary_output_file=demo_output,
            calendar_file=CALENDAR_FILE,
        )
        test_rows.append(
            {
                "Input Date": date,
                "Status": result.status,
                "DTD": result.dtd,
                "Was Written": result.wrote_output,
                "Message": result.message,
            }
        )

    # Deliberately skip 20251216 and request 20251217 directly.
    try:
        process_daily_dtd(
            "20251217",
            confirmed_file=CONFIRMED_FILE,
            temporary_input_file=demo_input,
            temporary_output_file=demo_output,
            calendar_file=CALENDAR_FILE,
        )
    except MissingTradingDayError as error:
        test_rows.append(
            {
                "Input Date": "20251217",
                "Status": "ERROR_MISSING_TRADING_DAY",
                "DTD": np.nan,
                "Was Written": False,
                "Message": str(error),
            }
        )

    for date in ("20251216", "20251217"):
        result = process_daily_dtd(
            date,
            confirmed_file=CONFIRMED_FILE,
            temporary_input_file=demo_input,
            temporary_output_file=demo_output,
            calendar_file=CALENDAR_FILE,
        )
        test_rows.append(
            {
                "Input Date": date,
                "Status": result.status,
                "DTD": result.dtd,
                "Was Written": result.wrote_output,
                "Message": result.message,
            }
        )

    demo_temporary_output = read_temporary_output(demo_output)

test_results = pd.DataFrame(test_rows)
display(test_results)
print("Final Temporary Output:")
display(demo_temporary_output)

## 4 Expected interpretation

- 20251213 and 20251214: weekend closures; skipped without writing Output.
- 20251215: the next Hong Kong trading day after the 12th; calculated and written.
- First request for 20251217: raises an error because 20251216 is incomplete.
- 20251216: calculated and written.
- Second request for 20251217: the sequence is now continuous, so it is calculated and written.

Final Temporary Output should contain only 20251215, 20251216, and 20251217.